## Colab session configuration

## Reading hg38 genome assembly 

Set parent directory.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import pathlib
from pathlib import Path
from functools import reduce

# Set data directory and base directory
base_dir = Path.cwd()

Download the hg38 fasta zipped file and unzip to standard fasta file

In [11]:
!mkdir -p data
!curl -L -C - "https://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz" -o data/hg38.fa.gz
!curl -L -C - "https://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.chrom.sizes" -o data/hg38.chrom.sizes
!gunzip -c data/hg38.fa.gz > data/hg38.fa

** Resuming transfer from byte position 983659424
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   314    0   314    0     0    677      0 --:--:-- --:--:-- --:--:--   676
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 11672  100 11672    0     0  59500      0 --:--:-- --:--:-- --:--:-- 59551


Try loading the unzipped fasta file using pyfaidx.

In [28]:
!pip install pyfaidx
from pyfaidx import Fasta

# Data directory
data_dir = base_dir / "data"
file_path = data_dir / "hg38.fa"

# Open fasta using pyfaidx
f = Fasta(data_dir / "hg38.fa")

# Open the chromsizes using pandas
c = pd.read_table(data_dir / "hg38.chrom.sizes", sep = "\t", index_col = 0, header = None)
c.columns = ["length"]

https://pypi.org/project/pyfaidx/

In [34]:
c

,length
0,
chr1,248956422
chr2,242193529
chr3,198295559
chr4,190214555
chr5,181538259
...,...
chrUn_KI270539v1,993
chrUn_KI270385v1,990
chrUn_KI270423v1,981


In [ ]:
f["chr1"][5000000:5000100]

>chr1:5000001-5000100
TCTTTGgtgtctgtgtgtgtgcgcgcgcacgtgcatgtgtgtgagttgtcattgtttttgtttgtTTTGTATTCCTCAGAAATAGAAAGCTCCCATTAGG

Now, we need to write a script that can draw random samples from the genome. We can outline this in pseudocode.

In [ ]:
rng = np.random.default_rng(111)

def sample_from_fasta(fasta_file, chrom_size_file, n_seqs, min_length, max_length):
    """
    Function to sample randomly from .fa genome

    Args:
        fasta_file      : Path to .fa file containing genome
        chrom_size_file : Path to tab-separate text file containing sequence names and sizes
        n_seqs          : Number of sequences to sample
        min_length      : Minimum sequence length to sample
        max_length      : Maximum sequence length to sample
    
    Returns:
        seqs : List of sampled sequences as strings
    """
    # Store file using pyfaidx
    f = Fasta(fasta_file)

    # Store length of each chromosome key in fasta
    chromosomes = f.keys()
    size_df = pd.read_table(chrom_size_file, sep = "\t", index_col = 0, header = None)
    size_df.columns = ["length"]

    # Normalize to obtain sampling weights for each chromosome
    weights = size_df["length"].to_numpy() / size_df["length"].to_numpy().sum()

    # Check that the fasta chromsomes match the size_df index***

    # Then, order them correctly

    # Initiate list to store sequences
    seqs = []

    # Number of sequences to sample
    for i in range(n_seqs):

        # Weighted sampling of chromosome
        chrom = rng.choice(chromosomes, p = lengths_df["length"].to_)
        length =  # store length of that chromsome

        # Randomly generate a starting index
        start_idx = rng.integers(0, length)

        # Randomly generate a sequence length
        seq_length = rng.integers(min_length, max_length + 1)

        # Slice the chromosome to obtain sequence
        seq = f[chrom][start_idx:start_idx + seq_length]
        seq = str(seq)

        # Split across "N" character if exists and take first substr
        seq = seq.split("N")[0]

        # Convert to uppercase, move non-alpha and non-ACGT characters 
        seq = "".join(char.strip().upper() for char in seq if char.isalpha() and char in "ACGT")

        # Check that sequences is still above minimum length
        if len(seq) >= min_length:
            seqs.append(seq)

    return seqs

Function to open and sample from the zipped fasta file.

## Model

In [ ]:
!pip install accelerate
!pip install datasets
!pip install transformers
!pip install torch
!pip install flash-attn

import accelerate
import flash_attn
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

torch.backends.cudnn.benchmark=True

Implement a sparse attention transformer.

Implement the model with 10 transformer blocks.

In [ ]:
class miniglm(nn.Module):

    def __init__(self):